In [ ]:
import matplotlib.pyplot as plt

# Keep consistent styling for figures
plt.rcParams.update({
    'font.family': 'Arial', 
    'font.size': 14,
    'axes.linewidth': 1.5,
    'axes.labelsize': 14,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'lines.linewidth': 2,
    'legend.fontsize': 12,
    'figure.dpi': 600,
})


#### Density Profile
density.xvy is generated by GROMACS build-in functions by running  
`gmx density -f md_center.xtc -n index.ndx -s md.tpr -o density.xvg -b 500000 -d Z -dens mass -ng 2 -sl 50`

In [ ]:
import numpy as np

# === File load and data extraction ===
file = './density.xvg'
data = np.loadtxt(file, comments=['@', '#'])
dz = data[:, 0]
water = data[:, 2]
popc = data[:, 1]

# === Plot ===
fig, ax = plt.subplots(figsize=(6, 8))
ax.plot(water, dz, '-', color='C0', alpha=1.0, label='Water')
ax.plot(popc, dz, '-', color='C1', alpha=1.0, label='POPC')
ax.set_xlabel('Density (kg/m$^3$)')
ax.set_ylabel('dZ (nm)')
ax.set_title('Density Profile')
ax.legend(loc='best', frameon=True)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('density_profiles.png')
plt.show()

#### Contact Frequency
1. pre-processed by [GetContacts](https://getcontacts.github.io/)  
`get_dynamic_contacts.py --topology ref.pdb --trajectory MD.xtc --itypes all --output contacts_D_H.tsv --sele "group 1" --sele2 "group 2` 
`get_contact_frequencies.py --input_files contacts_1_2.tsv --output_file resfrequencies_1_2.tsv`
2. post-processing TVSs
+ Convert 3-letter residue codes to 1-letter (optional).
+ Average chain-pair TSVs within a repeat; filter + heatmap.
+ Average across repeats; filter + heatmap.

In [ ]:
import csv
import re
from collections import defaultdict
import pandas as pd
import seaborn as sns

# === Convert three-letter codes to one-letter ===
def convert_residue_name(residue): # HSD is a common variant of HIS in PDB files
    residue_map = {
        'ASN':'N','GLY':'G','TYR':'Y','ASP':'D','SER':'S',
        'ALA':'A','ARG':'R','CYS':'C','GLN':'Q','GLU':'E',
        'HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M',
        'PHE':'F','PRO':'P','THR':'T','TRP':'W','VAL':'V','HSD':'H'
    } 
    return residue_map.get(residue, residue)

# === Extract numeric index from residue label ===
def residue_sort_key(label):
    match = re.search(r'(\d+)', label)
    return int(match.group(1)) if match else 0

# === Process TSV: strip chains D/H, convert to one-letter + index ===
def process_tsv(input_file, output_file):
    with open(input_file, 'r') as tsv_in, open(output_file, 'w', newline='') as tsv_out:
        reader = csv.reader(tsv_in, delimiter='\t')
        writer = csv.writer(tsv_out, delimiter='\t')
        for row in reader:
            # strip old chain prefixes
            row = [item.replace('a:', '') if item.startswith('a:') else item for item in row]
            row = [item.replace('b:', '') if item.startswith('b:') else item for item in row]
            new_row = []
            for item in row:
                if ':' in item:
                    parts = item.split(':')
                    one_letter = convert_residue_name(parts[-2])
                    idx = parts[-1]
                    new_row.append(f"{one_letter}{idx}")
                else:
                    new_row.append(item)
            writer.writerow(new_row)

# === Parameters ===
runs = ['R1', 'R2', 'R3']
base_dir = '/Users/chenyifang/Desktop/VMP1_ATG2'
threshold = 0.2  # filter out low frequencies for clearer visualization
# storage for averaging
global_freqs = defaultdict(list)

# === Plot each repeat ===
for run in runs:
    raw_fn = f"{base_dir}/{run}/resfrequencies_{run[1]}.tsv"
    conv_fn = f"{base_dir}/{run}/converted_{run[1]}.tsv"
    process_tsv(raw_fn, conv_fn)
    df = pd.read_csv(conv_fn, sep='\t', header=None,
                     names=['res1', 'res2', 'freq'], comment='#')
    df['res1'] = df['res1'].str.strip() 
    df['res2'] = df['res2'].str.strip()
    df = df[df['freq'] >= threshold]

    # accumulate for average
    for _, row in df.iterrows():
        global_freqs[(row.res1, row.res2)].append(row.freq)

    # fill missing as zero
    pivot = df.pivot(index='res1', columns='res2', values='freq').fillna(0)

    # --- Sort both axes by residue number ---
    pivot = pivot.loc[sorted(pivot.index, key=residue_sort_key),
                      sorted(pivot.columns, key=residue_sort_key)]

    n_rows, n_cols = pivot.shape
    fig_w = max(8, n_cols * 0.4 + 2.5)
    fig_h = max(6, n_rows * 0.4 + 1.5)

    # plot
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    heatmap = sns.heatmap(
        pivot,
        cmap='Blues',
        vmin=0,
        vmax=1,
        square=True,
        linewidths=0.2,
        linecolor='black',
        cbar_kws={'label': 'Contact Frequency', 'shrink':0.6}, 
        ax=ax
    )
    cbar = heatmap.collections[0].colorbar
    cbar.outline.set_linewidth(0.2)
    ax.set_title(f'Contact Frequency {run}', pad=10)
    ax.set_xlabel(r'$\mathrm{ATG2}^{\mathrm{NR}}$')
    ax.set_ylabel('VMP1')
    ax.set_xticks([i + 0.5 for i in range(n_cols)])
    ax.set_xticklabels(pivot.columns.tolist(), rotation=45, ha='right')
    ax.set_yticks([i + 0.5 for i in range(n_rows)])
    ax.set_yticklabels(pivot.index.tolist(), rotation=0)
    plt.tight_layout()
    plt.savefig(f"{base_dir}/contact_frequency_{run}.png")
    plt.savefig(f"{base_dir}/contact_frequency_{run}.svg")
    plt.show()

# === Compute and plot average ===
avg_data = [(r1, r2, sum(vals)/len(vals)) for (r1, r2 ), vals in global_freqs.items()]
avg_df = pd.DataFrame(avg_data, columns=['res1', 'res2', 'avg_freq'])
avg_df = avg_df[avg_df['avg_freq'] >= threshold]

# --- Save averaged data to CSV ---
avg_df.sort_values(by=['res1', 'res2'],
                   key=lambda col: col.map(residue_sort_key)
                   ).to_csv(f"{base_dir}/contact_frequency_average.csv", index=False)
pivot_avg = avg_df.pivot(index='res1', columns='res2', values='avg_freq').fillna(0)
pivot_avg = pivot_avg.loc[sorted(pivot_avg.index, key=residue_sort_key),
                          sorted(pivot_avg.columns, key=residue_sort_key)]

n_rows_avg, n_cols_avg = pivot_avg.shape
fig_w_avg = max(8, n_cols_avg * 0.4 + 2.5)
fig_h_avg = max(6, n_rows_avg * 0.4 + 1.5)

fig, ax = plt.subplots(figsize=(fig_w_avg, fig_h_avg))
heatmap_avg=sns.heatmap(
    pivot_avg,
    cmap='Blues',
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.2,
    linecolor='black',
    cbar_kws={'label': 'Contact Frequency', 'shrink': 0.7},
    ax=ax
)
cbar = heatmap_avg.collections[0].colorbar    
cbar.outline.set_linewidth(0.2)
ax.set_title('Average Contact Frequency', pad=10)
ax.set_xlabel(r'$\mathrm{ATG2}^{\mathrm{NR}}$')
ax.set_ylabel('VMP1')
ax.set_xticks([i + 0.5 for i in range(n_cols_avg)])
ax.set_xticklabels(pivot_avg.columns.tolist(), rotation=45, ha='right')
ax.set_yticks([i + 0.5 for i in range(n_rows_avg)])
ax.set_yticklabels(pivot_avg.index.tolist(), rotation=0)

plt.tight_layout()
plt.savefig(f"{base_dir}/contact_frequency_average_new.png")
plt.savefig(f"{base_dir}/contact_frequency_average_new.svg")
plt.show()

#### Calculation of Scrambled Lipids
1. Track the z-axis positions of POPC phosphate beads on a frame-by-frame basis using [Scramblyzer](https://github.com/Ladme/scramblyzer) or using PLUMED
2. Post-processing steps:  
+ For raw z-coordinate of lipid $j$ at frame $i$  
$$
\text{mid}_{i} = (\text{max}_{j}\mathrm{z}_{i,\,j} + \text{min}_{j}\mathrm{z}_{i,\,j})/2
$$
+ Then it recenters every lipid $$ \mathrm{z}^{'}_{i,\,j} = \mathrm{z}_{i,\,j} - \text{mid}_{i} $$
+ Leaflet assignment (+1 is upper leaflet while -1 is lower leaflet)  
$$
s_{i\,j} = \begin{cases}
+\,1 & \mathrm{z}^{'}_{i,\,j} > 0 \\
-\,1 & \mathrm{z}^{'}_{i,\,j} < 0 \end{cases}$$
+ Dwell-time–confirmed sign change: 
Define a confirmed leaflet state $S_{t}\in\{+1\,-1\}$ for lipid j. The first seen on the opposite side at a started time t0, $ s_{j}(t_{0}) = -S_{j}(t_{0})$. And it continously stays at the opposite side at time t, $s_{j}(t) = s_{j}(t_{0})$ and $t-t_{0} \ge \tau$. Then update the confirmed state is $S_{j}(t) = -S_{j}(t_{0})$.


In [ ]:
import numpy as np
# load
data = np.loadtxt('/Users/chenyifang/Desktop/RA/VMP1_Martini3_trjs/RUN1/positions.xvg', comments=['@','#'])
times = data[:,0]
zs_all = data[:,1:]
n_frames, n_lipids = zs_all.shape

def recenter(zs):
    mid = 0.5*(zs.max() + zs.min())
    return zs - mid

# initialize per‐lipid trackers
state = np.sign(recenter(zs_all[0]))  # confirmed leaflet ±1
candidate = np.zeros(n_lipids, dtype=int)  # 0 means “no candidate”
t0 = np.zeros(n_lipids)             # when candidate first seen
cumul = np.zeros(n_frames, int)
flips = 0

for i in range(1, n_frames):
    zs = recenter(zs_all[i])
    sign_now = np.where(zs > 0, 1, -1)

    for j in range(n_lipids):
        if sign_now[j] == state[j]:
            candidate[j] = 0
        elif candidate[j] == 0:
            candidate[j] = sign_now[j]
            t0[j] = times[i]
        elif sign_now[j] != candidate[j]:
            candidate[j] = 0
        else:
            if times[i] - t0[j] >= 10.0: #dwell time threshold
                state[j] = candidate[j]
                candidate[j] = 0
                flips += 1
    cumul[i] = flips

# plot
times_us = times / 1000.0
plt.plot(times_us, cumul, lw=2)
plt.xlabel('simulation time (us)')
plt.ylabel('number of scrambled lipids')
plt.title('scrambled lipids')
plt.tight_layout()
plt.show()

#### Cavity Volume Approximation
1. Take centers of mass (COMs) of the three helices: $C_{1}(t),\,C_{2}(t),\,C_{3}(t)$. Each is a 3D vector (x, y, z)
2. Compute a 2D triangle area in the membrane plane (xy) from those three points
3. Compute a height from how spread out they are in z.
4. Volume proxy = (triangle area)*(z-spread)

In [ ]:
import MDAnalysis as mda
import numpy as np
from pathlib import Path
from scipy.ndimage import uniform_filter1d

helix1_residues = "10-25"
helix2_residues = "130-140"
helix3_residues = "150-165"
target_sel_str = "resid 407 and resname POPE and name P"
smooth_window  = 50
FIRST_NS       = 1000.0

def extract_volume_trace(top, traj, first_ns=FIRST_NS):
    u = mda.Universe(top, traj)
    h1 = u.select_atoms(f"protein and resid {helix1_residues} and backbone")
    h2 = u.select_atoms(f"protein and resid {helix2_residues} and backbone")
    h3 = u.select_atoms(f"protein and resid {helix3_residues} and backbone")
    t_ns, volumes = [], []
    for i, ts in enumerate(u.trajectory):
        t_ps = getattr(ts, "time", None)
        t = t_ps / 1000.0
        if t > first_ns:
            break
        c1, c2, c3 = h1.center_of_mass(), h2.center_of_mass(), h3.center_of_mass()
        d12 = np.linalg.norm(c1[:2] - c2[:2])
        d13 = np.linalg.norm(c1[:2] - c3[:2])
        d23 = np.linalg.norm(c2[:2] - c3[:2])
        s   = (d12 + d13 + d23) / 2.0
        area = np.sqrt(max(s * (s - d12) * (s - d13) * (s - d23), 0.0))
        z_spread = max(c1[2], c2[2], c3[2]) - min(c1[2], c2[2], c3[2])
        t_ns.append(t)
        volumes.append(area * z_spread / 1000.0)
    return np.asarray(t_ns), np.asarray(volumes)

#data processing and plotting
top = "./md_center.gro"
traj = "./md_center.xtc"
t_ns, volumes = extract_volume_trace(top, traj)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_ns, volumes, color='C0', alpha=0.8, label='RUN1')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Volume (nm$^3$)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

#### Tilt Angle Calculation (Protein alongside with membrane) 
NOTE: it would be more suitable for rod-like proteins, like ATG2 protein in our case
Protein’s principal (long) axis from the mass-weighted inertia tensor
1. Select protein and recenter $\mathbf{r}'_i = \mathbf{r}_i - \mathbf{r}_{COM}$
2. Build the mass-weighted inertia tensor
$$ \mathbf{I} = \sum_{i} m_i \left[ \left(\mathbf{r}'_i \cdot \mathbf{r}'_i\right)\mathbf{I}_3 \;-\; \mathbf{r}'_i \left(\mathbf{r}'_i\right)^{\mathsf T} \right] $$ 
Eigen-decompose gives $\mathbf{I}v_k\,=\,\lambda_k v_k$ and for a rod-like object, the axis along the rod has the smallest moment of inertia $\mathbf{v}_0$  
3. Compute tilt angle relative to membrane normal
Membrane normal is assumed to be the unit z vector $\mathbf{n} = (0,\,0,\,1)$ and angle between axis and normal is then $\cos\theta=\mathbf{v}_0 \mathbf{n}$


In [ ]:
def calculate_tilt_angle(universe, protein_selection):
    
    protein = universe.select_atoms(protein_selection)
    coords = protein.positions - protein.center_of_mass()
    
    # Calculate moment of inertia tensor
    I = np.zeros((3, 3))
    for atom_pos, atom_mass in zip(coords, protein.masses):
        I += atom_mass * (np.dot(atom_pos, atom_pos) * np.eye(3) - 
                          np.outer(atom_pos, atom_pos))
    
    # Get principal axes (eigenvectors sorted by eigenvalues)
    _, eigenvectors = np.linalg.eigh(I)
    # Longest axis corresponds to smallest eigenvalue (for rod shape)
    principal_axis = eigenvectors[:, 0]
    # Membrane normal (z-axis)
    membrane_normal = np.array([0, 0, 1])
    # Calculate angle (use abs to get 0-90° range)
    cos_angle = np.dot(principal_axis, membrane_normal)
    angle = np.arccos(np.abs(cos_angle)) * 180 / np.pi
    return angle

u = mda.Universe("./md_center.gro", "./md_center.xtc")
angles = []
for ts in u.trajectory: 
    angle = calculate_tilt_angle(u, "protein")
    angles.append(angle)
# Plot
plt.figure(figsize=(10, 5))
plt.plot(angles)
plt.xlabel("Time")
plt.ylabel("Tilt angle")
plt.title("Protein tilt relative to membrane")
plt.tight_layout()
plt.show()